Thai Duc Minh DO's ECS784P Coursework 1 - QMUL 24/25

**NOTE: THE CODE UP UNTIL BEFORE THE "PROCESS THE DATA" SECTION WAS TAKEN FROM
 THIS LINK, BEFORE IT WAS MODIFIED TO SERVE THIS PROJECT.**
 https://www.kaggle.com/code/gimunu/transfermarkt-data-scraping

## Config

In [ ]:
# Original scraper's link:
# https://www.kaggle.com/code/gimunu/transfermarkt-data-scraping

N_LEAGUES = 5 # (EN, ES, DE, IT, FR)
BASE_URL = "https://www.transfermarkt.co.uk"
LEAGUES_URL = BASE_URL + "/wettbewerbe/europa/wettbewerbe"
CURRENT_YEAR = 25 #2024-2025 season
N_SEASON_HISTORY = 1

DELAY_BETWEEN_QUERIES = 0.5 #min delay in seconds spacing http queries
USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 14_7_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Safari/605.1.15'
MAX_WORKERS = 7 #thread pool max size

## Data classes and parsing logic

In [ ]:
# @title
import logging
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import namedtuple
from itertools import groupby
from operator import itemgetter
from datetime import datetime

from tqdm import tqdm

logger = logging.Logger(__name__)

date_format = "%b %d, %Y"
date_str1 = "Jun 30, 2025" #end of contract year

class Leagues:
    def __init__(self, scraper):
        self.scraper = scraper
        self.parseLeagues()

    def parseLeagues(self):
        leagues_soup = self.scraper.getSoup(LEAGUES_URL)
        LeagueTables = leagues_soup.find("table", class_="items").find("tbody")
        Leagues = LeagueTables.find_all("a", href=re.compile(r"wettbewerb/[A-Z]{1,2}1"), title=re.compile(r"\w"))
        Leagues = Leagues[:N_LEAGUES]
        LeagueUrlDic = {league.text: league["href"] for league in Leagues}
        logger.info("About to scrap leagues:", tuple(LeagueUrlDic.keys()))
        self.leaguesData = []
        for leagueName, leagueUrl in (pbar := tqdm(LeagueUrlDic.items())):
            pbar.set_description(f"{leagueName:<30}")
            self.leaguesData.append(League(leagueName, leagueUrl, self.scraper))

    def export(self, filename="StrikerPerformance.csv"):
        playerProfiles = [player.playerData for league in self.leaguesData for team in league.teamsData for player in team.playersData]
        df = pd.DataFrame(playerProfiles)
        df = df[sorted(df.columns, reverse=True)]
        goal_columns = [col for col in df if col.endswith("goals")]
        decreasing_total_goals = df[goal_columns].sum(axis=1).sort_values(ascending=False).index
        df.loc[decreasing_total_goals].to_csv(filename, index=False)


class League:
    def __init__(self, name, url, scraper):
        self.leagueName = name
        self.leagueSoup = scraper(url)
        self.scraper = scraper
        self.parseLeague()

    def parseLeague(self):
        teamsTable = self.leagueSoup.find("table", class_="items")
        teamUrls = teamsTable.find_all("td", class_="hauptlink no-border-links")
        teamUrls = [item.find("a") for item in teamUrls]
        teamUrls = {teamUrl.text.lstrip().rstrip(): teamUrl["href"] for teamUrl in teamUrls}
        self.teamsData = []
        for teamName, teamUrl in (pbar := tqdm(teamUrls.items(), leave=False)):
            pbar.set_description(f"{teamName:<30}")
            self.teamsData.append(Team(teamUrl, self.leagueName, self.scraper))


PATTERN_PLAYER_POS = re.compile(r"zentriert rueckennummer")
PATTERN_PLAYER_ATTRIBUTE = re.compile(r"info-table__content")


class Team:
    def __init__(self, url, name, scraper):
        self.leagueName = name
        self.url = url
        self.scraper = scraper
        self.parseTeam()

    def parseTeam(self):
        teamSoup = self.scraper(self.url)
        playerTable = teamSoup.find("table", class_="items")
        players = playerTable.find_all("td", class_="hauptlink")[::2]

        positions = [item["title"].lower() for item in playerTable.find_all("td", class_=PATTERN_PLAYER_POS)]

        excludedPositions = ["goalkeeper", "defender", "midfield"]

        isNotAttacker = [item["title"].lower() in excludedPositions for item in playerTable.find_all("td", class_=PATTERN_PLAYER_POS)]
        assert len(isNotAttacker) == len(players), "Mismatch between position and player data."

        players = [player for isNotAttack, player in zip(isNotAttacker, players) if not isNotAttack]
        players = {item.find("a").text.lstrip().rstrip(): item.find("a")["href"] for item in players}
        self.playersData = []

        def instanciate(cls, *args):
            try:
                return cls(*args)
            except Exception as exc:
                logger.error("Error in thread: `%s`", exc)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_url = {
                executor.submit(instanciate, PlayerProfile, playerName, playerUrl, self.scraper): playerUrl
                for playerName, playerUrl in players.items()
            }
            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    NewPlayerProfile = future.result()
                    NewPlayerProfile.playerData["current league"] = self.leagueName
                    self.playersData.append(NewPlayerProfile)
                except Exception as exc:
                    print("%r generated an exception: %s" % (url, exc))


perfRow = namedtuple("PerformanceRow", ["year", "games_played", "minutes_played", "goals_scored", "assists"])


class PlayerProfile:
    def __init__(self, playerName, playerUrl, scraper):
        self.playerUrl = playerUrl
        self.urlPerfPage = playerUrl.replace("profil", "leistungsdatendetails")
        self.scraper = scraper
        self.playerName = playerName
        self.parsePlayer()

    def parsePlayer(self):
        soup = self.scraper(self.playerUrl)

        # scraping profile page information
        playerAttributes = {}
        playerAttributes["name"] = self.playerName
        playerAttributes["ID"] = int(self.playerUrl.split("/")[-1]) #ID on transfermarkt

        StoredAttributes = {"Date of birth/Age:", "Position:", "Current club:", "Contract expires:"}
        entries = soup.find_all("span", class_=PATTERN_PLAYER_ATTRIBUTE)
        for key, val in zip(entries[::2], entries[1::2]):
            key = key.text.strip()
            val = val.text.strip()
            if key in StoredAttributes:
                if key == "Date of birth/Age:":
                    age = re.search(r'\((\d+)\)', val)  # Extracts digits inside parentheses
                    if age:
                        number = int(age.group(1))
                        playerAttributes["age"] = number
                elif key == "Contract expires:":
                    if val == "-":
                        playerAttributes["Remaining contract length (days)"] = "N/A"
                    else:
                        date_str2 = val
                        date_next_contract_year = datetime.strptime(date_str1, date_format)
                        date2_end_of_contract = datetime.strptime(date_str2, date_format)
                        difference = date2_end_of_contract - date_next_contract_year
                        playerAttributes["Remaining contract length (days)"] = difference.days
                else:
                    playerAttributes[key[:-1].lower()] = val.strip()

        if "position" in playerAttributes:
            playerAttributes["position"] = playerAttributes["position"].split("-")[-1].lstrip()

        # Extract all instances of 'waehrung' class which has the market value
        waehrung_elements = soup.find_all(class_="waehrung")
        if (waehrung_elements):
            market_value = waehrung_elements[0].find_next_sibling(string=True).strip()
            if (market_value):
                currency = waehrung_elements[0].text.strip()  # First element (€, $, etc.)
                multiplier = waehrung_elements[1].text.strip()  # Second element (k, M, etc.)
                market_value = market_value.strip()
                try:
                    market_value = float(market_value)
                    if multiplier.lower() == "k":
                        market_value *= 1_000
                    elif multiplier.lower() == "m":
                        market_value *= 1_000_000
                except ValueError:
                    market_value = None
                playerAttributes["Market Value (Euro)"] = market_value

        # scraping performance page information
        soup = self.scraper(self.urlPerfPage)
        performanceColumns = ("season", "games", "minutes", "goals", "assists")
        performanceRows = []
        for row in soup.find("div", class_="responsive-table").find("tbody").find_all("tr"):
            perfRow = PlayerProfile.parsePerformanceRow(row)
            if int(perfRow.year[:2]) < CURRENT_YEAR - N_SEASON_HISTORY:
                break
            performanceRows.append(perfRow)

        # performanceRows = agg(performanceRows, by='season', agg=sum)
        performanceDF = pd.DataFrame(data=performanceRows, columns=performanceColumns).groupby("season", sort=False).sum()
        performanceSeries = {"%s %s" % (row, col): performanceDF[col][row] for row in performanceDF.index for col in performanceDF.columns}

        self.playerData = playerAttributes | performanceSeries
        logger.info("\t%s done", self.playerData["name"])

    @staticmethod
    def parsePerformanceRow(row):
        cells = row.find_all("td")
        cells = list(map(lambda x: x.text.replace("\xa0", " ").strip(), cells))
        year, *_, games_played, goals_scored, assists, _, minutes_played = cells
        if re.match(r"\d{4}", year):
            year = int(year[2:])
            year = "%02d/%02d" % (year - 1, year)
        games_played = int(games_played) if games_played != "-" else 0
        goals_scored = int(goals_scored) if goals_scored != "-" else 0
        assists = int(assists) if assists != "-" else 0
        minutes_played = int(minutes_played[:-1].replace(".", "")) if minutes_played != "-" else 0
        return perfRow(year, games_played, minutes_played, goals_scored, assists)


In [ ]:
# @title
import sqlite3
from time import sleep, time
from threading import Lock

!pip install lz4
import lz4
import requests
from bs4 import BeautifulSoup
from lz4.frame import compress, decompress


cache_db = sqlite3.connect("page.db", check_same_thread=False)

class PageScraper:
    def __init__(self, cache_cb):
        self.lastQuery = -666
        self.cache_db = cache_db
        self.db_mutex = Lock()
        with self.db_mutex, self.cache_db:
            self.cache_db.execute("CREATE TABLE IF NOT EXISTS pages (url text PRIMARY KEY, content text)")

    def getSoup(self, url):
        if not url.startswith(BASE_URL):
            url = BASE_URL + url
        with self.db_mutex, self.cache_db:
            hit = self.cache_db.execute("select content from pages where url = ? limit 1;", (url,)).fetchone()
        if hit:
            content = decompress(hit[0])
        else:
            waiting_time = self.lastQuery + DELAY_BETWEEN_QUERIES - time()
            if waiting_time > 0:
                sleep(waiting_time)
            req = requests.get(url, headers={"User-agent": USER_AGENT})
            req.raise_for_status()
            self.lastQuery = time()
            content = req.content
            with self.db_mutex, self.cache_db:
                self.cache_db.execute("insert into pages (url, content) VALUES (?, ?)", (url, compress(content)))
        return BeautifulSoup(content, "lxml")

    def __call__(self, url):
        return self.getSoup(url)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.2 MB/s eta 0:00:00


## Let the scrap begin

In [ ]:
#uncomment out this section if need to gather data
#scraper = PageScraper(cache_db)
#data = Leagues(scraper)
#data.export('scoring_performance.csv')

## Process the data

In [ ]:
# We first should convert the output .csv file into a Pandas frame
raw_df = pd.read_csv('scoring_performance.csv')

# Next we should remove attributes that won't contribute to the model, such as player name & ID (too sample-specific)
# and current club because each club only has a few forwards, samples per club too small so we do samples per league
raw_df.drop(columns=['name', 'ID', 'current club'], inplace=True)

# Filter out players w/ NaN in any of their attributes (incomplete info)
raw_df = raw_df.dropna()
# Remove players with negative contract days due to unknown circumstances around them
raw_df = raw_df[raw_df["Remaining contract length (days)"] >= 0]

raw_df = raw_df.reset_index(drop=True)

# Reorganize column order for ease of reading
new_order = ['age', 'position', 'current league', 'Market Value (Euro)', 'Remaining contract length (days)', '24/25 games', '24/25 minutes', '24/25 goals', '24/25 assists']
raw_df = raw_df[new_order]

print(raw_df)

In [ ]:
# Categorize league & positions into numbers since we can't fit strings as predictors into models
positions = {'Forward': 1, 'Left Winger': 2, 'Right Winger': 3, 'Second Striker': 4}
leagues = {'Premier League': 1, 'LaLiga': 2, 'Bundesliga': 3, 'Serie A': 4, 'Ligue 1': 5}

raw_df['position'] = raw_df['position'].replace(positions).astype(int)
raw_df['current league'] = raw_df['current league'].replace(leagues).astype(int)

columns_to_convert = ['Market Value (Euro)', 'Remaining contract length (days)',
                    '24/25 games', '24/25 minutes',
                    '24/25 goals', '24/25 assists']

raw_df[columns_to_convert] = raw_df[columns_to_convert].astype(int)

# Separate our label from predictors
X = raw_df.drop(columns=['Market Value (Euro)'])
y = raw_df['Market Value (Euro)']

print(X)

***************************************************************************************

START

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def draw_scatter(y, y_pred):
    plt.figure(figsize=(8, 8))
    plt.scatter(y, y_pred, alpha=0.6, color='teal')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    plt.grid(True)
    plt.title('Predicted vs. Actual Market Values')
    plt.xlabel('Actual Market Value (Euro)')
    plt.ylabel('Predicted Market Value (Euro)')
    plt.show()

# Evaluate pipeline using cross validation to generate predictions while checking for overfitting
def evaluate_pipeline(pipeline, X, y, cv=None, model_name="Model", random_state=25, check_overfitting=True):
    if cv is None:
        cv = KFold(n_splits=10, shuffle=True, random_state=random_state)

    y_pred_cv = cross_val_predict(pipeline, X, y, cv=cv)
    draw_scatter(y, y_pred_cv)

    cv_r2 = r2_score(y, y_pred_cv)
    cv_mse = mean_squared_error(y, y_pred_cv)
    cv_rmse = cv_mse ** 0.5
    cv_mae = mean_absolute_error(y, y_pred_cv)

    print(f"{model_name} - CV Results:")
    print(f"R²:  {cv_r2:.4f}")
    print(f"RMSE: {cv_rmse:,.2f}")
    print(f"MAE: {cv_mae:,.2f}")
    print("-" * 40)

    results = {"CV_R²": cv_r2, "CV_RMSE": cv_rmse, "CV_MAE": cv_mae}

    if check_overfitting:
        pipeline.fit(X, y)
        y_pred_train = pipeline.predict(X)
        # Calculate training metrics
        train_r2 = r2_score(y, y_pred_train)
        train_mse = mean_squared_error(y, y_pred_train)
        train_rmse = train_mse ** 0.5
        train_mae = mean_absolute_error(y, y_pred_train)

        print(f"{model_name} - Training Results:")
        print(f"R²:  {train_r2:.4f}")
        print(f"RMSE: {train_rmse:,.2f}")
        print(f"MAE: {train_mae:,.2f}")
        print("-" * 40)

        results.update({"Train_R²": train_r2, "Train_RMSE": train_rmse, "Train_MAE": train_mae})

    return results

# Define pipelines for LR and RF
pipeline_lr = Pipeline([
    ('lr', LinearRegression())
])

pipeline_rf = Pipeline([
    ('rf', RandomForestRegressor(
        n_estimators=250,
        max_depth=25))
])

# ATTEMPT 1: Evaluate pipelines with dataset as is
cv = KFold(n_splits=10, shuffle=True, random_state=25)
metrics_lr = evaluate_pipeline(pipeline_lr, X, y, model_name="Linear Regression")
metrics_rf = evaluate_pipeline(pipeline_rf, X, y, model_name="Random Forest")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

# Check feature importance for RF
def evaluate_feature_importance(pipeline, X, title="RF Feature Importances"):
    try:
        check_is_fitted(pipeline.named_steps['rf'])
    except NotFittedError as e:
         raise NotFittedError("RandomForestRegressor in the pipeline is not fitted") from e

    ## Extract feature importances
    importances = pipeline.named_steps['rf'].feature_importances_

    try:
        features = X.columns
    except AttributeError:
        features = [f"Feature_{i}" for i in range(X.shape[1])]

    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(8, 6))
    plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
    plt.xlabel('Importance')
    plt.ylabel('Features')
    plt.title(title)
    plt.gca().invert_yaxis()  # Highest importance at the top
    plt.show()

    return importance_df

pipeline_rf.fit(X, y)
importance_df = evaluate_feature_importance(pipeline_rf, X)
print(importance_df)

In [ ]:
# @title
# Remove "position" predictor due to its lack of importance. This improved the models' performance
X = X.drop('position', axis=1)

# ATTEMPT 2: Apply additional processing to dataset

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# Preprocessing steps:
# 1. StandardScaler
# 2. Log transformation
# 3. Re-encode categorical variables

# Define feature groups to apply different normalization forms
numeric_features = ['age', '24/25 minutes', '24/25 games', '24/25 assists', '24/25 goals', 'Remaining contract length (days)']
skewed_features = ['age', '24/25 minutes', '24/25 games', '24/25 assists', '24/25 goals',]
categorical_features = ['current league']

rf_n_estimators = 150
rf_max_depth = 15

log_transformer = FunctionTransformer(np.log1p, validate=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('log', log_transformer, skewed_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'  # Optionally pass through any remaining features
)

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr', LinearRegression())
])

pipeline_rf = Pipeline(steps=[
    #additional processing negatively affects RF performance
     #('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(
        random_state=25,
        n_estimators=rf_n_estimators,
        max_depth=rf_max_depth))
])

# ATTEMPT 2: w/ feature transformation
metrics_lr = evaluate_pipeline(pipeline_lr, X, y, model_name="Linear Regression")
metrics_rf = evaluate_pipeline(pipeline_rf, X, y, model_name="Random Forest")

For LR, the additional transformations (data normalisation & feature representation) lead to a more stable model.

Meanwhile for RF, hyperparameter tuning (n_estimators & max_depth) is essential and allow for better generalization as RF is generally less sensitive to scaling & log transformation.

However, despite all these processing, the model's performance barely improved. We could explore more advanced feature engineering / feature selection.

In [ ]:
# @title
#we are left with close to 630 samples

#consider drawing a graph to show the value variation between all 5 leagues
league_forward_counts = raw_df['current league'].value_counts()

# Calculate the total market value of forwards for each league
# Note that 1: Premier League, 2: LaLiga, 3: Bundesliga, 4: Serie A, 5: Ligue 1
league_values = raw_df.groupby('current league')['Market Value (Euro)'].sum()
print(league_values)

# Create a bar plot for total market values by league
plt.figure(figsize=(10, 5))
league_values.plot(kind='bar', color='skyblue')
plt.xlabel('Current League')
plt.ylabel('Total Forward Market Value (Euro)')
plt.title('Total Forward Market Value by League')
plt.xticks(rotation=45)
plt.show()

#draw a graph here for visualization of e.g forward distribution, total market value

********************************************************************************

In [ ]:
# @title
# Attempt at simplifying predictors
# Combine goals & assists to become goal contributions
# Change league values from categorial to weight via MinMaxScaler (based on sum of forwards' market value in each league)

#Premier League total market value: 3655875000
#LaLiga total market value: 1980800000
#Bundesliga total market value: 1294225000
#Serie A total market value: 1569900000
#Ligue 1 total market value: 1181200000

sum_forwards_value = 3655875000 + 1980800000 + 1294225000 + 1569900000 + 1181200000

epl_val_percent = round(3655875000 / sum_forwards_value * 100, 2)
laliga_val_percent = round(1980800000 / sum_forwards_value * 100, 2)
bund_val_percent = round(1294225000 / sum_forwards_value * 100, 2)
seriea_val_percent = round(1569900000 / sum_forwards_value * 100, 2)
ligue1_val_percent = round(1181200000 / sum_forwards_value * 100, 2)

leagues_val = np.array([
    [epl_val_percent],
    [laliga_val_percent],
    [bund_val_percent],
    [seriea_val_percent],
    [ligue1_val_percent]
])

# Uncomment below if need to get each league's percentage in total forward market vale
#print(epl_val_percent, laliga_val_percent, bund_val_percent, seriea_val_percent, ligue1_val_percent)
from sklearn.preprocessing import MinMaxScaler

# Close approx of each leagues' portion in the sum of forward market value in top 5 EU leagues
# e.g. if 3655 = 1 then 1980 should be approx 54% of 3655, so using a feature range of 0.32 - 1 is best for clear representation)
scaler = MinMaxScaler(feature_range=(0.32, 1))
normalized_vals = scaler.fit_transform(leagues_val)

replace_league_vals = {1: normalized_vals[0],
                       2: normalized_vals[1],
                       3: normalized_vals[2],
                       4: normalized_vals[3],
                       5: normalized_vals[4]}

X2 = X
X2['current league'] = X2['current league'].replace(replace_league_vals)

X2['24/25 goals/assists'] = X2['24/25 goals'] + X2['24/25 assists']
X2 = X2.drop(['24/25 goals', '24/25 assists'], axis=1)

#final dataset
#print(X2)

In [ ]:
numeric_features2 = ['age', '24/25 minutes', '24/25 games', '24/25 goals/assists' , 'Remaining contract length (days)']
skewed_features2 = ['age', '24/25 minutes', '24/25 games', '24/25 goals/assists', ]

#estimators 124, maxdepth 6, min samples leaf 2, maxleafnodes 27 give best performance

rf_n_estimators2 = 124
rf_max_depth2 = 6

log_transformer2 = FunctionTransformer(np.log1p, validate=True)

preprocessor2 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features2),
        ('log', log_transformer2, skewed_features2)
    ],
    remainder='passthrough'  # Optionally pass through any remaining features
)

pipeline_lr2 = Pipeline(steps=[
    ('preprocessor2', preprocessor2),
    ('lr2', LinearRegression())
])

# Hyperparams fine tuning
pipeline_rf2 = Pipeline(steps=[
    # additional processing affects RF performance
    #('preprocessor2', preprocessor2),
    ('rf2', RandomForestRegressor(random_state=25,
                                 n_estimators=rf_n_estimators2,
                                 max_depth=rf_max_depth2,
                                 min_samples_leaf=2, #2 best
                                 max_leaf_nodes=27, #27 best val for reducing overfitting
                                 ))

])

metrics_lr2 = evaluate_pipeline(pipeline_lr2, X2, y, model_name="Linear Regression")
metrics_rf2 = evaluate_pipeline(pipeline_rf2, X2, y, model_name="Random Forest")

There was some performance gain after further feature engineering & hyperparameter tuning, but the RF model overfits overall. The LR model's performance was subpar, implying it underfit, and is too simple to understand the data's complexity.

Why low overall performance?
- Data Limitations: variability in the data, measurement errors or the domain complexity (like sports performance) might cap the maximum achievable predictive accuracy.
- Model Capacity and Complexity: LR and RF limitations might not be able to capture enough nuances
